In [1]:
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core import StorageContext
from llama_index.graph_stores.neo4j import Neo4jGraphStore


from llama_index.llms.openai import OpenAI
from IPython.display import Markdown, display

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# from fastapi import FastAPI
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=1024, context_window = 2048, num_output=256))


# storage_context = StorageContext.from_defaults(persist_dir="mini512")
# index = load_index_from_storage(storage_context)

[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_2732\2831050717.py:26: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=1024, context_window = 2048, num_output=256))


In [2]:
import os
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = "test"

graph_store = Neo4jGraphStore(
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    url=NEO4J_URI,
    database=NEO4J_DATABASE,
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [3]:
# Clean dataset first
ew = graph_store.query(
    """
MATCH (n) detach delete n
"""
)
file_path = r"48-77gptout.txt"
documents = SimpleDirectoryReader(input_files = [file_path]).load_data()
print(len(documents))
print(f"Document Metadata: {documents[0].metadata}")
storage_context = StorageContext.from_defaults(graph_store=graph_store)

1
Document Metadata: {'file_path': '48-77gptout.txt', 'file_name': '48-77gptout.txt', 'file_type': 'text/plain', 'file_size': 7299, 'creation_date': '2024-11-08', 'last_modified_date': '2025-01-31'}


In [4]:
from llama_index.core.prompts.base import PromptTemplate
from llama_index.core.prompts.prompt_type import PromptType
DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW = (
    "Some text from ancient story Ramayan is provided below. Given the text, extract knowledge graph triplets with the following rules:\n"

    "1. Extract triplets with the following patterns:\n"
    "   (charecter, relation, charecter) - Resolve the relations between charecters based on the context given\n"
    "   (charecter, context, event) - Resolve the relations between charecters and the major events in the context given\n"
    "   (charecter, 'is in', Canto) - Mapping for each charecter to canto\n"
    "2. Focus more on the important events and relations between charecters\n"
    "3. Avoid stopwords.\n"
    "4. use snake casing for everything in triplets\n"
    "5. for (charecter, relation, charecter) type of triplets, use the following relations:\n"
    "- is (mother/father/brother/sister/wife/husband/son/daughter/siblings/aunt/uncle/niece/nephew/teacher/mentor/related_in_canto_'number') of\n"
    "6. While generating triplets corresponding to charecter context event, focus only on important event of the given text, no need to split every sentence of text\n"
    "7. The number of triplets triplets of type (charecter, 'is in', Canto) should not exceed 10, and the number of other two type of triplets should not exceed {max_knowledge_triplets} \n"
    "These triplets are used for RAG, so only give important triplets and make sure there is no important data missing\n"
    "Text: {text}\n"
    "Triplets:\n"
)
DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW = PromptTemplate(
    DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW,
    prompt_type=PromptType.KNOWLEDGE_TRIPLET_EXTRACT,
)

In [5]:
index = KnowledgeGraphIndex.from_documents(
    documents,
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

Prompt:  Some text from ancient story Ramayan is provided below. Given the text, extract knowledge graph triplets with the following rules:
1. Extract triplets with the following patterns:
   (charecter, relation, charecter) - Resolve the relations between charecters based on the context given
   (charecter, context, event) - Resolve the relations between charecters and the major events in the context given
   (charecter, 'is in', Canto) - Mapping for each charecter to canto
2. Focus more on the important events and relations between charecters
3. Avoid stopwords.
4. use snake casing for everything in triplets
5. for (charecter, relation, charecter) type of triplets, use the following relations:
- is (mother/father/brother/sister/wife/husband/son/daughter/siblings/aunt/uncle/niece/nephew/teacher/mentor/related_in_canto_'number') of
6. While generating triplets corresponding to charecter context event, focus only on important event of the given text, no need to split every sentence of t

In [6]:
storage_context.persist()

In [9]:
query_engine = index.as_query_engine(
    include_text=False, response_mode="tree_summarize"
)

response = query_engine.query("How is viswamitra related to rama?")

Prompt:  A question is provided below. Given the question, extract up to 10 keywords from the text. Focus on extracting the keywords that we can use to best lookup answers to the question. Avoid stopwords.
---------------------
How is viswamitra related to rama?
---------------------
Provide keywords in the following comma-separated format: 'KEYWORDS: <keywords>'

KEYWORDS: viswamitra, rama, relationship

Prompt:  Context information from multiple sources is below.
---------------------
kg_schema: {'schema': 'Node properties are the following:\n\nRelationship properties are the following:\n\nThe relationships are the following:\n'}

The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
['RELATED_IN_CANTO_26_42', 'Lakshmana', 'CONTEXT_CANTO_1_16', 'Supporting_rama']
['RELATED_IN_CANTO_26_42', 'Lakshmana', 'RELATED_IN_CANTO_26_42', 'Viswamitra']
['RELATED_IN_CANTO_26_42', 'Vis

In [10]:
response.response

"Viswamitra is related to Rama in that they are both present in Canto 26, Verse 42, as indicated by the knowledge sequence.  Additionally, Vasistha, who is related to Viswamitra in Canto 32, Verse 48, is also present with Rama and Viswamitra in Canto 1, Verse 16.  Lakshmana, Rama's brother, is also related to Viswamitra in Canto 26, Verse 42.  This suggests a shared narrative context or interaction between Viswamitra and Rama within these specific sections of the text.\n"

In [19]:
storage_context1 = StorageContext.from_defaults(persist_dir="storage")

In [20]:
index = load_index_from_storage(storage_context1)

In [ ]:
query_engine = index.as_query_engine(
    include_text=False, response_mode="tree_summarize"
)

response = query_engine.query("Who is the father of Rama?")

In [ ]:
print(response.response)